<a href="https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Capstone Research Paper: Data-Driven Content Prioritization Engine

## 1. Research Question
* **Core Question:** Can a machine learning model, trained on anonymized search console performance data, out-perform simple heuristic rules in prioritizing underperforming URLs for SEO editorial action?
* **Decision Supported:** Assists editorial and SEO teams in prioritizing content refresh workflows, title/meta rewrites, and internal linking strategies across high-impression, low-CTR web pages.

## 2. Dataset Overview & Public Safety
* **Release & Source:** `FlyRank/internship-warehouse` (`fact_content_daily_performance` table).
* **Scope:** March 2026 daily snapshot window (~9.8M aggregated daily performance rows).
* **Exclusions & Public-Safety:** All client domains, raw URLs, and search query strings are salted and pseudonymized (`client_hash_id`, `content_hash_id`). No private or client-identifying information is included or exposed.

## 3. Methodology & Validation Design
* **Features Used:** `gsc_impressions`, `gsc_position`, `ga4_sessions`, `session_ctr`.
* **Label Definition:** Binary classification target ($1$ if the content item generated $\ge 1$ organic search click, else $0$).
* **Baseline Rule:** Priority Score formula targeting striking-distance URLs (Positions 4–20):
  $$\text{Priority Score} = \left(\frac{\text{gsc\_impressions}}{\text{gsc\_position} + 1}\right) \times (1 - \text{CTR})$$
* **Validation Strategy:** Grouped split by `client_hash_id` (80% train / 20% validation) to evaluate generalization on unseen client websites without domain-level leakage.

In [2]:
import duckdb
import os
import json
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Authenticate DuckDB with Hugging Face Token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

data_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Load aggregated dataset
df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        AVG(gsc_avg_position) AS gsc_position,
        SUM(ga4_sessions) AS ga4_sessions,
        CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(ga4_sessions) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS session_ctr,
        ((SUM(gsc_impressions) * 1.0 / (AVG(gsc_avg_position) + 1)) * (1 - (CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END))) AS baseline_rule_score,
        CASE WHEN SUM(gsc_clicks) > 0 THEN 1 ELSE 0 END AS target
    FROM read_parquet('{data_path}')
    GROUP BY client_hash_id, content_hash_id
""").df().fillna(0)

print(f"Dataset successfully loaded: {len(df)} aggregated content records.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset successfully loaded: 331437 aggregated content records.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [3]:
# Train-Test Split Grouped by Client ID
features = ['gsc_impressions', 'gsc_position', 'ga4_sessions', 'session_ctr']
X = df[features]
y = df['target']
groups = df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
baseline_val = df['baseline_rule_score'].iloc[val_idx]

# Fit Random Forest
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)

# Compare AUC Scores
rf_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
baseline_auc = roc_auc_score(y_val, baseline_val)

results_table = pd.DataFrame({
    'Approach': ['Week 4 Rule-Based Baseline', 'Week 5 Random Forest ML'],
    'Validation Strategy': ['Grouped by Client ID', 'Grouped by Client ID'],
    'ROC-AUC Score': [round(baseline_auc, 4), round(rf_auc, 4)]
})

print("=== RESULTS: MODEL VS BASELINE ===")
print(results_table.to_string(index=False))

=== RESULTS: MODEL VS BASELINE ===
                  Approach  Validation Strategy  ROC-AUC Score
Week 4 Rule-Based Baseline Grouped by Client ID         0.9597
   Week 5 Random Forest ML Grouped by Client ID         0.9701


## 5. Limitations & Honest Framing
* **Non-Causal:** The model identifies statistical correlations between impression volume, ranking position, and click engagement. It does not guarantee that updating a title tag will automatically increase rankings or CTR.
* **SERP Feature Noise:** Pages ranking in positions 1–5 that suffer from zero-click searches (e.g., featured snippets or knowledge panels) appear as false positives in the prioritization queue.
* **Batch Mode:** The model operates on aggregated monthly snapshots and does not reflect intra-week ranking volatility.

## 6. Content Action Playbook Recommendations
* **Primary Actions:**
  1. `REWRITE_TITLE_AND_META`: Flagged for URLs on Page 1/2 with high impressions but low relative CTR.
  2. `ADD_INTERNAL_LINKS`: Flagged for striking-distance URLs (Positions 11–20) to boost page authority.
  3. `REFRESH_CONTENT`: Flagged for legacy pages experiencing steady position decay.

* **Strict No-Go Policy:**
  * No automated page deletions or unpublishing.
  * No unreviewed AI metadata injection into production CMS.
  * No automated 301/302 URL redirects.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [4]:
# Generate Capstone Output Artifacts
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export Capstone Summary Metrics JSON
capstone_metrics = {
    "dataset_total_records": int(len(df)),
    "baseline_roc_auc": float(round(baseline_auc, 4)),
    "random_forest_roc_auc": float(round(rf_auc, 4)),
    "data_source_credit": "Built on the FlyRank ML Internship dataset (https://flyrank.ai)",
    "status": "CAPSTONE_COMPLETE"
}

with open('work/outputs/capstone_metrics.json', 'w') as f:
    json.dump(capstone_metrics, f, indent=4)

print("Capstone artifacts successfully generated.")
print("Metrics JSON:", capstone_metrics)

Capstone artifacts successfully generated.
Metrics JSON: {'dataset_total_records': 331437, 'baseline_roc_auc': 0.9597, 'random_forest_roc_auc': 0.9701, 'data_source_credit': 'Built on the FlyRank ML Internship dataset (https://flyrank.ai)', 'status': 'CAPSTONE_COMPLETE'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
